In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

print("drive exists:", Path("/content/drive").is_dir())
print("MyDrive exists:", Path("/content/drive/MyDrive").is_dir())

In [ ]:
# ============================================================
# RESTORE FEVER FIT / VALIDATION SPLIT
# NO RETRIEVAL / NO ENCODING / NO FAISS
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive")

# 找所有可能的 FEVER split 檔
candidates = []

patterns = [
    "*fever*split*.csv",
    "*FEVER*split*.csv",
    "*split*.csv",
]

for pat in patterns:
    for p in ROOT.rglob(pat):
        try:
            df = pd.read_csv(p)
        except Exception:
            continue

        cols = {c.lower(): c for c in df.columns}

        # 需要至少有 query_id + split
        qcol = None
        scol = None

        for k in ["query_id", "query-id", "qid"]:
            if k in cols:
                qcol = cols[k]
                break

        for k in ["split", "partition", "subset"]:
            if k in cols:
                scol = cols[k]
                break

        if qcol and scol:
            vals = set(df[scol].astype(str).str.lower().unique())

            if (
                any(v in vals for v in ["fit", "train"])
                and any(v in vals for v in ["validation", "val", "dev"])
            ):
                candidates.append((p, df, qcol, scol))

print("candidate split files:", len(candidates))

for i, (p, df, qcol, scol) in enumerate(candidates[:20]):
    print(f"\n[{i}] {p}")
    print(" rows:", len(df))
    print(" split values:", sorted(df[scol].astype(str).unique())[:20])

if not candidates:
    raise FileNotFoundError(
        "No FEVER fit/validation split CSV found in mounted MyDrive."
    )

In [ ]:
# ============================================================
# VERIFY + RESTORE FEVER v0.13 FIT / VALIDATION SPLIT
# ============================================================

import pandas as pd
import hashlib
from pathlib import Path

SPLIT_PATH = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/"
    "fever-boundary-external-replication-v013/"
    "20260817-151852/"
    "v013_boundary_query_split.csv"
)

assert SPLIT_PATH.is_file(), SPLIT_PATH

df = pd.read_csv(SPLIT_PATH)

print("columns:", df.columns.tolist())
print("rows:", len(df))
print(df["split"].value_counts())

qcol = next(
    c for c in ["query_id", "query-id", "qid"]
    if c in df.columns
)

FIT_IDS = (
    df.loc[df["split"].astype(str).str.lower() == "fit", qcol]
    .astype(str)
    .tolist()
)

VAL_IDS = (
    df.loc[df["split"].astype(str).str.lower() == "validation", qcol]
    .astype(str)
    .tolist()
)

assert FIT_IDS
assert VAL_IDS
assert set(FIT_IDS).isdisjoint(VAL_IDS)
assert len(FIT_IDS) + len(VAL_IDS) == len(df)

def membership_sha(ids):
    return hashlib.sha256(
        "\n".join(sorted(map(str, ids))).encode("utf-8")
    ).hexdigest()

print("\n" + "=" * 72)
print("FEVER v0.13 SPLIT RESTORED")
print("=" * 72)
print("split file:", SPLIT_PATH)
print("FIT:", len(FIT_IDS))
print("VAL:", len(VAL_IDS))
print("FIT SHA:", membership_sha(FIT_IDS))
print("VAL SHA:", membership_sha(VAL_IDS))
print("overlap:", len(set(FIT_IDS) & set(VAL_IDS)))

In [ ]:
# ============================================================
# ARC-v0.25 — FEVER-E5 SEVERITY-MATCHED MECHANISM CALIBRATION
# RESTORE EXISTING V0.18 ARTIFACTS
#
# NO CORPUS ENCODING
# NO INDEX BUILD
# NO VALIDATION TRAJECTORIES
# FIT-ONLY SEVERITY SELECTION
# ============================================================

!pip -q install faiss-cpu

from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import hashlib
import json
import os

import faiss
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ============================================================
# 0. Frozen identities
# ============================================================

SEED = 20260824

RUN = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/"
    "cross-encoder-fever-replication-v018/"
    "20260819-015645"
)

SPLIT_PATH = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/"
    "fever-boundary-external-replication-v013/"
    "20260817-151852/"
    "v013_boundary_query_split.csv"
)

RAW_FEVER = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/"
    "raw-datasets/fever"
)

assert RUN.is_dir(), RUN
assert SPLIT_PATH.is_file(), SPLIT_PATH

DIM = 384
TOP_RETRIEVE = 100
UTILITY_K = 10

HIGH_NPROBE = 64

# Freeze candidate grid BEFORE calibration.
NPROBE_CANDIDATES = [
    1, 2, 4, 6, 8,
    12, 16, 24, 32
]

faiss.omp_set_num_threads(os.cpu_count() or 1)


# ============================================================
# 1. Helpers
# ============================================================

def membership_sha(ids):
    return hashlib.sha256(
        "\n".join(sorted(map(str, ids))).encode("utf-8")
    ).hexdigest()


def norm_vec(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    return x / max(float(np.linalg.norm(x)), eps)


def ndcg(ids, relevant, k=UTILITY_K):
    ids = np.asarray(ids, dtype=np.int64)[:k]

    gains = np.asarray(
        [
            1.0 if int(i) in relevant else 0.0
            for i in ids
        ],
        dtype=np.float64,
    )

    discounts = (
        1.0 /
        np.log2(np.arange(2, len(ids) + 2))
    )

    dcg = float(
        np.sum(gains * discounts)
    )

    m = min(k, len(relevant))

    if m == 0:
        return 0.0

    idcg = float(
        np.sum(
            1.0 /
            np.log2(np.arange(2, m + 2))
        )
    )

    return dcg / idcg


def search_index(index, q, nprobe):
    index.nprobe = int(nprobe)

    q = norm_vec(q)[None, :]

    scores, ids = index.search(
        q,
        TOP_RETRIEVE
    )

    valid = ids[0] >= 0

    return (
        scores[0][valid],
        ids[0][valid]
    )


# ============================================================
# 2. Restore and verify frozen split
# ============================================================

split_df = pd.read_csv(SPLIT_PATH)

FIT_IDS = (
    split_df.loc[
        split_df["split"].eq("fit"),
        "query_id"
    ]
    .astype(str)
    .tolist()
)

VAL_IDS = (
    split_df.loc[
        split_df["split"].eq("validation"),
        "query_id"
    ]
    .astype(str)
    .tolist()
)

FIT_SHA = membership_sha(FIT_IDS)
VAL_SHA = membership_sha(VAL_IDS)

EXPECTED_FIT_SHA = (
    "85c01de943636a080abe10cb18cf5538b"
    "705b89f6f042a74c92bac366d89612c"
)

EXPECTED_VAL_SHA = (
    "5e7bd8e3e5e3f0120b1e93726a191832"
    "85624c405ef68b34005c8766a9568b42"
)

assert len(FIT_IDS) == 3350
assert len(VAL_IDS) == 3316

assert FIT_SHA == EXPECTED_FIT_SHA
assert VAL_SHA == EXPECTED_VAL_SHA

assert set(FIT_IDS).isdisjoint(VAL_IDS)

print("Frozen split — PASS")
print("FIT:", len(FIT_IDS))
print("VAL:", len(VAL_IDS))


# ============================================================
# 3. Verify v0.18 protocol provenance
# ============================================================

protocol_path = (
    RUN /
    "v018_cross_encoder_protocol.json"
)

protocol = json.loads(
    protocol_path.read_text()
)

assert protocol["encoder"] == (
    "intfloat/e5-small-v2"
)

assert (
    protocol["fit_membership_sha256"]
    == FIT_SHA
)

assert (
    protocol["validation_membership_sha256"]
    == VAL_SHA
)

print("\nv0.18 protocol alignment — PASS")


# ============================================================
# 4. Restore query IDs + embeddings
# ============================================================

QUERY_IDS_PATH = (
    RUN /
    "dev_query_ids.txt"
)

QUERY_EMB_PATH = (
    RUN /
    "dev_query_embeddings.float32.npy"
)

DEV_QUERY_IDS = (
    QUERY_IDS_PATH
    .read_text()
    .splitlines()
)

dev_query_embeddings = np.load(
    QUERY_EMB_PATH,
    mmap_mode="r"
)

assert len(DEV_QUERY_IDS) == 6666

assert dev_query_embeddings.shape == (
    6666,
    DIM
)

DEV_QUERY_INDEX = {
    str(qid): i
    for i, qid
    in enumerate(DEV_QUERY_IDS)
}

assert all(
    q in DEV_QUERY_INDEX
    for q in FIT_IDS
)

assert all(
    q in DEV_QUERY_INDEX
    for q in VAL_IDS
)

print(
    "Query embeddings:",
    dev_query_embeddings.shape
)

print("Query alignment — PASS")


# ============================================================
# 5. Restore existing FAISS indexes
# ============================================================

PQ_PATH = (
    RUN /
    "fever-e5-small-v2-ivfpq-"
    "nlist4096-m32-nbits8.faiss"
)

SQ_PATH = (
    RUN /
    "fever-e5-small-v2-"
    "ivfsq8-nlist4096.faiss"
)

assert PQ_PATH.is_file()
assert SQ_PATH.is_file()

print("\nLoading PQ32...")
pq32 = faiss.read_index(
    str(PQ_PATH)
)

print("Loading SQ8...")
sq8 = faiss.read_index(
    str(SQ_PATH)
)

EXPECTED_DOCS = 5_416_568

assert pq32.ntotal == EXPECTED_DOCS
assert sq8.ntotal == EXPECTED_DOCS

print(
    "PQ32 ntotal:",
    f"{pq32.ntotal:,}"
)

print(
    "SQ8 ntotal:",
    f"{sq8.ntotal:,}"
)

print("FAISS restore — PASS")


# ============================================================
# 6. Find FEVER qrels automatically
# ============================================================

QRELS_ROOT = (
    RAW_FEVER /
    "qrels"
)

assert QRELS_ROOT.is_dir(), QRELS_ROOT

qrel_files = [
    p
    for p in QRELS_ROOT.rglob("*")
    if p.is_file()
]

print("\nQrels files:")

for p in qrel_files:
    print(" ", p)


def try_read_qrels(path):
    # Try TSV first, then generic CSV.
    attempts = [
        {"sep": "\t"},
        {"sep": ","},
        {"sep": None, "engine": "python"},
    ]

    for kw in attempts:
        try:
            x = pd.read_csv(
                path,
                **kw
            )

            if len(x.columns) >= 2:
                return x

        except Exception:
            pass

    return None


qrel_candidates = []

for p in qrel_files:

    if p.suffix.lower() not in {
        ".tsv",
        ".csv",
        ".txt"
    }:
        continue

    x = try_read_qrels(p)

    if x is None:
        continue

    cols_lower = {
        c.lower(): c
        for c in x.columns
    }

    qcol = next(
        (
            cols_lower[c]
            for c in [
                "query-id",
                "query_id",
                "qid",
            ]
            if c in cols_lower
        ),
        None
    )

    dcol = next(
        (
            cols_lower[c]
            for c in [
                "corpus-id",
                "corpus_id",
                "doc_id",
                "document_id",
            ]
            if c in cols_lower
        ),
        None
    )

    scol = next(
        (
            cols_lower[c]
            for c in [
                "score",
                "relevance",
                "rel",
            ]
            if c in cols_lower
        ),
        None
    )

    if qcol is None or dcol is None:
        continue

    x[qcol] = x[qcol].astype(str)
    x[dcol] = x[dcol].astype(str)

    coverage = len(
        set(x[qcol])
        &
        set(DEV_QUERY_IDS)
    )

    qrel_candidates.append(
        {
            "path": p,
            "df": x,
            "qcol": qcol,
            "dcol": dcol,
            "scol": scol,
            "coverage": coverage,
        }
    )


assert qrel_candidates, (
    "No readable FEVER qrels found."
)

qrel_candidates.sort(
    key=lambda z: z["coverage"],
    reverse=True
)

print(
    "\nCandidate qrels coverage:"
)

for c in qrel_candidates:
    print(
        c["coverage"],
        "|",
        c["path"]
    )


# Use all qrel files that overlap
# frozen 6666 queries.
usable = [
    c
    for c in qrel_candidates
    if c["coverage"] > 0
]

assert usable


# ============================================================
# 7. Combine positive qrels for frozen 6666 queries
# ============================================================

qrel_parts = []

for c in usable:

    x = c["df"].copy()

    qcol = c["qcol"]
    dcol = c["dcol"]
    scol = c["scol"]

    x = x[
        x[qcol].isin(DEV_QUERY_IDS)
    ].copy()

    if scol is not None:
        x[scol] = pd.to_numeric(
            x[scol],
            errors="coerce"
        )

        x = x[
            x[scol] > 0
        ].copy()

    qrel_parts.append(
        x[[qcol, dcol]]
        .rename(
            columns={
                qcol: "query_id",
                dcol: "doc_id",
            }
        )
    )

qrels_raw = (
    pd.concat(
        qrel_parts,
        ignore_index=True
    )
    .drop_duplicates()
)

print(
    "\nPositive qrel pairs:",
    f"{len(qrels_raw):,}"
)

covered_queries = set(
    qrels_raw["query_id"]
)

missing_q = (
    set(DEV_QUERY_IDS)
    -
    covered_queries
)

print(
    "Queries with qrels:",
    len(covered_queries)
)

print(
    "Frozen queries missing qrels:",
    len(missing_q)
)

assert len(missing_q) == 0, (
    list(sorted(missing_q))[:20]
)


# ============================================================
# 8. Map qrel document IDs to exact v0.18 FAISS rows
#
# IMPORTANT:
# scan corpus_doc_ids.txt once;
# keep only qrels-relevant doc IDs.
# ============================================================

CORPUS_IDS_PATH = (
    RUN /
    "corpus_doc_ids.txt"
)

assert CORPUS_IDS_PATH.is_file()

needed_doc_ids = set(
    qrels_raw["doc_id"]
    .astype(str)
)

print(
    "\nRelevant corpus IDs needed:",
    len(needed_doc_ids)
)

doc_to_row = {}

with open(
    CORPUS_IDS_PATH,
    "r",
    encoding="utf-8"
) as f:

    for row, line in enumerate(
        tqdm(
            f,
            total=EXPECTED_DOCS,
            desc="Mapping qrel docs"
        )
    ):

        doc_id = line.rstrip("\n")

        if doc_id in needed_doc_ids:
            doc_to_row[doc_id] = row


missing_docs = (
    needed_doc_ids
    -
    set(doc_to_row)
)

print(
    "Mapped relevant docs:",
    len(doc_to_row)
)

print(
    "Missing relevant docs:",
    len(missing_docs)
)

assert not missing_docs, (
    list(missing_docs)[:20]
)


# ============================================================
# 9. Construct QRELS in FAISS row space
# ============================================================

QRELS = defaultdict(set)

for r in qrels_raw.itertuples(
    index=False
):
    QRELS[str(r.query_id)].add(
        int(
            doc_to_row[
                str(r.doc_id)
            ]
        )
    )

assert all(
    q in QRELS
    and len(QRELS[q]) > 0
    for q in FIT_IDS
)

assert all(
    q in QRELS
    and len(QRELS[q]) > 0
    for q in VAL_IDS
)

print("\nQrels row alignment — PASS")


# ============================================================
# 10. Reproduce existing v0.18 FIT one-shot
# BEFORE severity search.
# ============================================================

fit_baseline_rows = []

for qid in tqdm(
    FIT_IDS,
    desc="Reproducing v0.18 FIT baseline"
):

    q = dev_query_embeddings[
        DEV_QUERY_INDEX[qid]
    ]

    _, pq_ids = search_index(
        pq32,
        q,
        64
    )

    _, sq_ids = search_index(
        sq8,
        q,
        64
    )

    rel = QRELS[qid]

    fit_baseline_rows.append(
        {
            "query_id": qid,
            "pq32_n64":
                ndcg(pq_ids, rel),
            "sq8_n64":
                ndcg(sq_ids, rel),
        }
    )

fit_base = pd.DataFrame(
    fit_baseline_rows
)

PQ_MEAN = float(
    fit_base["pq32_n64"].mean()
)

SQ_MEAN = float(
    fit_base["sq8_n64"].mean()
)

REP_GAP = (
    SQ_MEAN
    -
    PQ_MEAN
)

print()
print("=" * 80)
print("V0.18 FIT BASELINE REPRODUCTION")
print("=" * 80)

print("PQ32 n64:", PQ_MEAN)
print("SQ8  n64:", SQ_MEAN)
print("REP gap  :", REP_GAP)


# v0.18 published all-dev-ish one-shot
# summary is 0.482381 / 0.737180.
# FIT need not be bit-identical to that
# aggregate, but should be close and sane.

assert 0.40 < PQ_MEAN < 0.55
assert 0.65 < SQ_MEAN < 0.80
assert REP_GAP > 0.15

print("Baseline semantic reproduction — PASS")


# ============================================================
# 11. FIT-only nprobe severity calibration
# ============================================================

cal_rows = []

for qid in tqdm(
    FIT_IDS,
    desc="FIT nprobe calibration"
):

    q = dev_query_embeddings[
        DEV_QUERY_INDEX[qid]
    ]

    rel = QRELS[qid]

    row = {
        "query_id": qid
    }

    # Common high condition:
    # SQ8 nprobe=64
    _, ids_hi = search_index(
        sq8,
        q,
        HIGH_NPROBE
    )

    row["sq8_n64"] = ndcg(
        ids_hi,
        rel
    )

    for np_low in NPROBE_CANDIDATES:

        _, ids_low = search_index(
            sq8,
            q,
            np_low
        )

        row[
            f"sq8_n{np_low}"
        ] = ndcg(
            ids_low,
            rel
        )

    cal_rows.append(row)


cal = pd.DataFrame(
    cal_rows
)


# ============================================================
# 12. Select severity-matched nprobe
# ============================================================

HIGH_MEAN = float(
    cal["sq8_n64"].mean()
)

match_rows = []

for np_low in NPROBE_CANDIDATES:

    low_mean = float(
        cal[
            f"sq8_n{np_low}"
        ].mean()
    )

    search_gap = (
        HIGH_MEAN
        -
        low_mean
    )

    mismatch = abs(
        search_gap
        -
        REP_GAP
    )

    match_rows.append(
        {
            "nprobe_low":
                np_low,

            "search_low_ndcg10":
                low_mean,

            "high_ndcg10":
                HIGH_MEAN,

            "search_gap":
                search_gap,

            "representation_gap":
                REP_GAP,

            "absolute_gap_mismatch":
                mismatch,

            "relative_mismatch":
                (
                    mismatch
                    /
                    max(
                        abs(REP_GAP),
                        1e-12
                    )
                ),
        }
    )


match_df = (
    pd.DataFrame(match_rows)
    .sort_values(
        [
            "absolute_gap_mismatch",
            "nprobe_low",
        ]
    )
    .reset_index(drop=True)
)

print()
print("=" * 80)
print("FIT SEVERITY MATCH TABLE")
print("=" * 80)

display(
    match_df.round(6)
)


BEST = match_df.iloc[0]

MATCHED_NPROBE = int(
    BEST["nprobe_low"]
)

MATCHED_SEARCH_GAP = float(
    BEST["search_gap"]
)

ABS_MISMATCH = float(
    BEST["absolute_gap_mismatch"]
)

REL_MISMATCH = float(
    BEST["relative_mismatch"]
)


print()
print("=" * 80)
print("FROZEN MATCH")
print("=" * 80)

print(
    "representation gap :",
    REP_GAP
)

print(
    "matched nprobe low :",
    MATCHED_NPROBE
)

print(
    "search-effort gap  :",
    MATCHED_SEARCH_GAP
)

print(
    "absolute mismatch  :",
    ABS_MISMATCH
)

print(
    "relative mismatch  :",
    REL_MISMATCH
)


# ============================================================
# 13. Freeze ARC-v0.25 protocol
# BEFORE validation trajectories.
# ============================================================

ARC_ROOT = Path(
    "/content/drive/MyDrive/"
    "rag-pq-checkpoints/arc-v0"
)

V025_ROOT = (
    ARC_ROOT /
    "fever-e5-severity-matched-mechanism-v025"
)

V025_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

RUN_ID = datetime.now(
    timezone.utc
).strftime(
    "%Y%m%d-%H%M%S"
)

OUT = (
    V025_ROOT /
    RUN_ID
)

OUT.mkdir(
    parents=True,
    exist_ok=False
)


fit_base.to_csv(
    OUT /
    "v025_fit_representation_baseline.csv",
    index=False
)

cal.to_csv(
    OUT /
    "v025_fit_nprobe_calibration.csv",
    index=False
)

match_df.to_csv(
    OUT /
    "v025_fit_severity_match_table.csv",
    index=False
)


PROTOCOL = {
    "status":
        "ARC_V025_SEVERITY_MATCH_FROZEN_BEFORE_VALIDATION_TRAJECTORIES",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "source_v018_run":
        str(RUN),

    "source_v018_protocol_sha256":
        hashlib.sha256(
            protocol_path.read_bytes()
        ).hexdigest(),

    "dataset":
        "FEVER",

    "encoder":
        "intfloat/e5-small-v2",

    "corpus_rows":
        EXPECTED_DOCS,

    "fit_query_count":
        len(FIT_IDS),

    "validation_query_count":
        len(VAL_IDS),

    "fit_membership_sha256":
        FIT_SHA,

    "validation_membership_sha256":
        VAL_SHA,

    "calibration_split":
        "FIT only",

    "validation_trajectory_accessed":
        False,

    "representation_low": {
        "index":
            "IVF-PQ32",
        "nprobe":
            64,
    },

    "common_high": {
        "index":
            "IVF-SQ8",
        "nprobe":
            64,
    },

    "search_effort_candidates":
        NPROBE_CANDIDATES,

    "selected_search_effort_low_nprobe":
        MATCHED_NPROBE,

    "fit_representation_gap":
        REP_GAP,

    "fit_matched_search_gap":
        MATCHED_SEARCH_GAP,

    "fit_absolute_gap_mismatch":
        ABS_MISMATCH,

    "fit_relative_gap_mismatch":
        REL_MISMATCH,

    "selection_rule":
        (
            "Choose nprobe_low minimizing "
            "|FIT nDCG@10 gap(SQ8@64-PQ32@64) "
            "- FIT nDCG@10 gap(SQ8@64-SQ8@nprobe_low)|"
        ),

    "post_selection_retuning_allowed":
        False,

    "negative_null_or_reversal_retained":
        True,
}


PROTOCOL_PATH = (
    OUT /
    "v025_frozen_severity_match_protocol.json"
)

PROTOCOL_PATH.write_text(
    json.dumps(
        PROTOCOL,
        indent=2,
        sort_keys=True
    )
)

PROTOCOL_SHA = hashlib.sha256(
    PROTOCOL_PATH.read_bytes()
).hexdigest()

(
    OUT /
    "V025_PROTOCOL_SHA256.txt"
).write_text(
    f"{PROTOCOL_SHA}  "
    f"{PROTOCOL_PATH.name}\n"
)

print()
print("=" * 80)
print("ARC-v0.25 FIT CALIBRATION COMPLETE")
print("=" * 80)

print("OUT:", OUT)

print(
    "Protocol SHA:",
    PROTOCOL_SHA
)

print(
    "MATCHED_NPROBE:",
    MATCHED_NPROBE
)

print()
print(
    "DO NOT CHANGE THE MATCH AFTER "
    "SEEING VALIDATION TRAJECTORIES."
)